**This notebook is used to load a file data as it is into the table**

In [1]:
%run ../utility/common_functions.ipynb

In [2]:
# Variable declaration
session = get_active_session()
session.sql("USE DATABASE CUSTOMER_BILLING_DB").collect()
session.sql("USE SCHEMA PUBLIC").collect()

run_start_time = datetime.now()
pipeline_name = "Customer Billing Pipeline"
task_name = "Stage 2 Load Data Into Bronze Layer"
run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

In [3]:
# Find summary file in extracted folder
final_unzip_path = unzip_folder.rstrip("/")
file_name_prefix = "summary"
summary_file_name, summary_file_path = read_recursive_file_data(final_unzip_path)
if not summary_file_name or not summary_file_path:
    raise FileNotFoundError(f"Summary file not found in {final_unzip_path}")
print("summary_file_name =>", summary_file_name)
print("summary_file_path =>", summary_file_path)

In [6]:
# List Bronze YAML configuration files from Snowflake stage
all_files = session.sql(f"LIST {bronze_yaml_path}").collect()
for actual_file_path in all_files:
    yaml_file_name = actual_file_path["name"].rsplit("/", 1)[-1]
    if not yaml_file_name.lower().endswith((".yaml", ".yml")):
        continue

    run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    absolute_path = bronze_yaml_path.rstrip("/") + "/" + yaml_file_name
    table_name = yaml_file_name.rsplit(".", 1)[0]

    try:
        # Read YAML configuration
        yaml_content = open_file_read_data(absolute_path)
        reading_yaml_data = yaml.safe_load(yaml_content)

        ytable_name = reading_yaml_data["table"]["name"]
        ytable_schema_name = reading_yaml_data["table"]["schema_name"]
        final_table_name = ytable_schema_name + "." + ytable_name
        ytable_source_file = reading_yaml_data["table"]["source_file"]
        ytable_columns_names = reading_yaml_data["table"]["columns"]

        # File name pattern
        file_name_prefix = ytable_source_file.split("*")[0]
        file_name_pattern = rf"^{file_name_prefix}_(\d{{8}})_(\d{{6}})\.csv$"

        # Recursive search
        final_file_name, final_file_path = read_recursive_file_data(final_unzip_path)
        print("final_file_name =>", final_file_name)
        print("final_file_path =>", final_file_path)

        if not final_file_name or not final_file_path:
            raise FileNotFoundError(f"No matching CSV file found for table: {table_name}")

        # START LOG
        process_log("Start",table_name=table_name,layer="BRONZE",file_name=final_file_name)

        # File name validation
        file_name_validation = match_file_name_pattern(file_name_pattern,final_file_name)
        print("file_name_validation =>", file_name_validation)

        # Read summary CSV from Snowflake stage
        df_summary = (session.read.options({"PARSE_HEADER": True,"INFER_SCHEMA": True}).csv(summary_file_path))

        # Trim summary column names
        df_summary = df_summary.to_df(*[c.strip() for c in df_summary.columns])

        # Get summary count for current CSV file
        final_file_name_without_ext = final_file_name.replace(".csv", "")
        summary_count_row = (df_summary.filter(col('"file_name"').contains(lit(final_file_name_without_ext))).select(col('"count"')).first())
        if summary_count_row is None:
            raise ValueError(f"File {final_file_name} not found in summary file")
        summary_count = summary_count_row[0]

        # Read business CSV from Snowflake stage
        df_final_file_name = (session.read.options({"PARSE_HEADER": True,"INFER_SCHEMA": True}).csv(final_file_path))
        table_count = df_final_file_name.count()
        print("summary_count =>", summary_count)
        print("table_count =>", table_count)

        # Validate record count
        if table_count == (int(summary_count) - 1):
            # Load data into Bronze table
            df_final_file_name.write.mode("overwrite").save_as_table(final_table_name)
            data_load_message = "Data load successfully"
            print("data_load_message =>", data_load_message)

            # Get total records inserted
            total_rows = session.table(final_table_name).count()
            print("total_inserted_count =>", total_rows)
            # SUCCESS LOG
            process_log("Success",table_name=table_name,layer="BRONZE",total_inserted_count=total_rows,file_name=final_file_name)
        else:
            data_load_message = ("Summary count is not matched hence process stopped")
            print("data_load_message =>", data_load_message)
            raise ValueError(data_load_message)
    except Exception as e:
        # FAILED LOG
        process_log("Failed",table_name=table_name,layer="BRONZE",error_message=str(e))
        raise